# ```Lakehouse``` Tutorial #

### Instalation ###
```deltalake``` can be installed via ```pip3 install deltalake```. We'll also need to install ```pip3 install pandas pyarrow``` to interact with ```DataFrames```.
```fastf1``` can be installed via ```pip3 install fastf1```.

### Documentation ###
- ```fastf1``` Repo - [Link](https://github.com/theOehrly/Fast-F1);
- ```fastf1``` Documentation - [Link](https://docs.fastf1.dev/);

### Importing Dependencies ###

In [1]:
import os
import uuid
import json
import pandas as pd
from datetime import datetime, timezone

import fastf1

from deltalake import write_deltalake, DeltaTable

import warnings
warnings.filterwarnings("ignore")
import logging 
logging.disable(logging.INFO) # 'fastf1' uses built-in 'logging' module
logging.disable(logging.WARNING)

### Lakehouse Foundations & Architecture ###
**Data Warehouses (1980s–2000s)**<br>
A ```data ```warehouse``` is a centralized repository for structured, relational data. It uses a ```schema-on-write model```, meaning *data must conform to predefined schema before it can be loaded*. This enforces data quality but creates rigidity (any schema change requires ETL pipeline modifications) for semi-structured/unstructured data.

```Warehouses``` typically tightly couple storage with compute, meaning you scale both together (which makes them expensive at scale). 

**Data Lakes (2010s)**<br>
A data ```lake``` *stores raw data in its native format* (JSON, Parquet, CSV, etc) on cheap object storage (S3, GCS, Azure). It uses a ```schema-on-read model```, meaning *data is stored as-is and schema is applied at query time*. This provides flexibility and low storage cost, but introduces no ACID transactions, no data consistency guarantees, no schema enforcement, and eventual data swamps (where data quality degrades over time because anyone can write anything).

**Lakehouse (2020s)**<br>
A ```lakehouse``` combines the cheap, open storage format of a ```lake``` with the transactional guarantees, schema enforcement, and management features of a ```warehouse```. The key innovation is the *open table format* — a metadata layer that sits on top of object storage files and provides ACID transactions, schema enforcement, time travel, and efficient upserts.

This architecture decouples 3 concerns:
- *Storage*: open formats (Parquet files + transaction logs) on commodity object storage/local filesystem;
- *Compute*: pluggable engines that can read the same data simultaneously (pandas, DuckDB, Spark, Trino, etc);
- *Catalog*: a metadata service that maps table names to their storage locations and tracks schema, partitions, and history;

### Open Table Formats ###
**Delta Lake**<br>
Created by ```Databricks```, ```Delta Lake``` *stores data as Parquet files with a transaction log* that records all changes as JSON entries. The log is the source of truth and allow readers to check which files are active (latest version). Delta Lake supports ACID transactions, schema enforcement/evolution, and ```MERGE```/```UPDATE```/```DELETE``` operations.

**Apache Iceberg**<br>
Created by ```Netflix```, ```Iceberg``` uses a more sophisticated metadata hierarchy: *a catalog entry points to a metadata file, which points to manifest lists, which point to manifests, which point to data files*. This multi-level indirection enables fast planning (figuring out which files to read) without scanning the entire table. 

**Apache Hudi**<br>
Created by ```Uber```, ```Hudi``` was designed for *incremental processing and streaming workloads*. It supports 2 table types: ```Copy-on-Write``` (similar to ```Delta```/```Iceberg```) and ```Merge-on-Read``` (which maintains a merge between base files and incremental log files for fast upserts). Its strength is in near-real-time ingestion with upsert-heavy workloads.

### Lakehouse Concepts ###
**ACID Transactions**<br>
In storage systems, a ```transaction``` is *any operation treated as a single unit of work, which either completes fully or does not complete at all*, and leaves the storage system in a consistent state. 

```ACID``` is an acronym that refers to 4 key properties that define a ```transaction```
- ```Atomicity```: each statement (read, write, update or delete) is treated as a single unit (either the entire statement is executed, or none of it). This prevents data loss and corruption;
- ```Consistency```: ensures that transactions only make changes to tables in predefined, predictable ways. This ensures that corruption or errors in data do not create unintended consequences for the tables' integrity;
- ```Isolation```: when multiple users are reading/writing from the same table at once, transaction isolation ensures that concurrency doesn't affect one another (even though they're occurring simultaneously);
- ```Durability```: ensures that changes to data made by successful transactions will be saved, even in the event of system failure;

Traditional DBs achieve ```ACID``` through *locking and write-ahead logs managed by a single process*. Open table formats achieve ```ACID``` on distributed storage through *optimistic concurrency control* (a writer creates new files, then atomically commits a log entry pointing to those files - if two writers conflict, the second one retries or fails; readers always see a consistent snapshot since they read the log to determine which files belong to the current version).

**Schema Enforcement vs Schema Evolution**<br>
*Schema enforcement* rejects writes that don't match the table schema. *Schema evolution* allows the schema to change over time (adding columns, widening types) while maintaining backward compatibility. 

**Time Travel**<br>
Every commit to a table creates a new version, meaning it's possible to read any historical version by version number/timestamp. This is useful for debugging and auditing.

**Partitioning**<br>
Tables can be partitioned by one or more columns. Partitioning *physically separates data into directories by partition value*, enabling ```partition pruning``` (query engine skips directories that can't contain matching rows).

**Catalog**<br>
A ```catalog``` is a *mapping from a table name to its physical storage path*. It also *stores schema metadata, partitioning configuration, and table properties*, providing centralized governance. This allows to reference tables by name rather than hardcoding paths.

### The Project ###
We'll build a local ```lakehouse``` with the following architecture:
- *Storage*: local filesystem with Parquet + Delta transaction logs;
- *Table format*: Delta Lake;
- *Compute*: Pandas for transformations, DuckDB for SQL queries and analytics workloads;
- *Catalog*: Python dictionary;
- *Data source*: ```FastF1``` API;
- *Data flow*: Raw → Bronze → Silver → Gold → Feature Store → ML

**Storage Layout**<br>
The ```lakehouse``` uses local filesystem with a clear directory hierarchy:

```
lakehouse/
├── raw/           # Raw API responses with metadata (JSON/Parquet)
├── bronze/        # Raw, append-only, data (Delta)
├── silver/        # Cleansed, conformed, deduplicated tables (Delta)
├── gold/          # Star schema fact/dimension tables (Delta)
├── feature_store/ # Feature group tables + metadata
├── cache/         # FastF1 API cache
└── _catalog.json  # Table name → path mapping
```

Each ```zone``` is a *directory containing one subdirectory per table*. ```Partitioning``` creates nested directories within each table directory.

### Medallion Architecture ###
We use the ```medallion``` architecture to organizes data into 3 layers of increasing quality and structure: ```bronze```, ```silver```, and ```gold```. Each layer has a clear purpose, strict rules about allowed transformations, and serves as input for the next layer.

**Separation of Concerns**<br>
The architecture can be defined as specific activities (*ingestion*, *data quality/cleansing*, and *business logic*) each being handled in separate layers. This separation makes it easier to debug, test, and modify (each layer becomes a ```checkpoint```).

**Bronze Layer (Landing Zone)**<br>
```Bronze``` layer answers the question: *what did the source give us, and when?*

- *Append-only or Upsert by key*: data is never deleted or overwritten (except via ```upsert``` for idempotency), it's meant preserves history;
- *Schema-on-read to schema-on-write transition*: applies a ```schema``` to raw data (shifting from ```JSON``` blobs), without (yet) enforcing business rules;
- *Metadata-rich*: every record carries ingestion metadata (fetch timestamp, source, pipeline run ID, batch ID);
- *Partitioned by ingestion batch*: typically partitioned by date (or equivalent);
- *No business logic*: no calculations, no joins, no filtering (it's the true source, just typed and stored efficiently);

**Silver Layer (Cleansed and Conformed Zone)**<br>
```Silver``` layer answers the question: *what is the clean, trustworthy representation of this data?*

- *Cleansed*: handles ```NULLs```, types cast correctly, invalid rows quarantined/fixed;
- *Deduplicated*: exactly one record per logical entity (ie one row per driver per lap per session);
- *Conformed*: column names follow standard convention, types are consistent across tables, ```Foreign Keys``` are validated;
- *Slowly Changing Dimensions (```SCDs```)*: if an entity changes over time (ie driver switches teams), layer tracks history using effective dates + flags (SCD Type 2) or overwrites (SCD Type 1);
- *Queryable*: tables are source-of-truth for data engineers (used if you need to understand the data);

**Gold Layer (Consumption Zone)**<br>
```Gold``` layer answers the question: *what does the business need to make decisions?*

- *Business logic*: aggregations, calculations, and ```JOINs``` that encode business rules;
- *Star schema*: ```fact``` tables (events, metrics) surrounded by ```dimension``` tables (entities);
- *Pre-computed*: common aggregates are materialized so consumers don't need to recompute them;
- *Denormalized for read performance*: tables may join ```facts```/```dimensions``` to avoid the user needing to do so;

**Data Quality Gates**<br>
Between each layer, there should be a ```quality gate``` (set of checks that data must pass before being promoted to next layer). Common checks include:
- *Completeness*: are there unexpected ```NULLs``` in required columns?
- *Uniqueness*: are ```Primary Keys``` actually unique?
- *Referential integrity*: do ```Foreign Keys``` in ```fact``` tables exist in dimension tables?
- *Range validation*: are numeric values within expected ranges?
- *Freshness*: is data from the expected time period?

If a check fails, the pipeline should either quarantine the offending records, fail loudly, or log a warning.

**Idempotency**<br>
A pipeline is ```idempotent``` if *running it multiple times produces the same result* as running it once (critical for reliability). In our case, we'll achieved it through Delta ```merge()``` operations: the pipeline checks if a record already exists (by ```Primary Key```) and either updates it or inserts it (never duplicating it). 

### Slowly Changing Dimensions (SCDs) ###
Dimensions change over time (```schema``` evolution): drivers might switch teams, circuits/countries might be renamed/changed, etc. The main strategies to handle this are:
- *SCD Type 0 (Retain Original)*: never change the dimension record (history is permanent);
- *SCD Type 1 (Overwrite)*: update the dimension record in place (dimension value is always the current value - simple, but loses history);
- *SCD Type 2 (Track History)*: create a new record for each change, with effective dates and a ```is_valid``` flag (this preserves full history at the cost of complexity);

In our example, ```dim_drivers``` uses ```SCD Type 2``` for team changes, ```dim_circuits``` uses ```SCD Type 1``` (circuit details rarely change), ```dim_constructors``` also uses ```SCD Type 1```.

### Star Schema ###
The ```star schema``` is the *canonical dimensional model* for analytical workloads and *consists of a central ```fact``` table surrounded by ```dimension``` tables*. The ```fact``` table holds measurable quantitative data and the ```dimensions``` hold descriptive context.

**Fact Tables**<br>
A ```fact``` table *contains measurements (metrics) at a specific grain* (declared explicitly). *Each row represents one observation* (event). They're typically narrow (few columns) but long (many rows) and are composed mostly of ```Foreign Keys``` (descriptive attributes belong in ```dimensions```) and numeric measures.

**Dimension Tables**<br>
A ```dimension``` table *contains the descriptive attributes* that give context to the facts. They are typically wide (many columns) but short (fewer rows) and shouldn't be normalized into sub-tables (flat, wide, dimension are faster to query).

### Schema Evolution ###
Below we structure how ```FastF1``` data flows through the medallion layers:

**Bronze Tables**<br>
| Table | Source | Grain (1 row per...) | Key Columns |
|---|---|---|---|
| ```bronze.sessions``` | Session metadata | Session | season, round, session_type |
| ```bronze.laps``` | Lap data | Driver-lap in a session | season, round, session_type, driver_number, lap_number |
| ```bronze.weather``` | Weather data | Weather reading | season, round, session_type, timestamp |
| ```bronze.results``` | Session results | Driver in a session | season, round, session_type, driver_number |

**Silver Tables**<br>
| Table | Transformations | Key Columns |
|---|---|---|
| ```silver.dim_drivers``` | Extract unique drivers, normalize names, add ```driver_id``` (PK) | driver_id (PK), driver_number, full_name, nationality |
| ```silver.dim_constructors``` | Extract unique constructors/teams | constructor_id (PK), constructor_name, nationality |
| ```silver.dim_circuits``` | Extract unique circuits | circuit_id (PK), circuit_name, country, city |
| ```silver.dim_sessions``` | Clean session metadata, link to circuits | session_id (PK), season, round, session_type, circuit_id (FK), date |
| ```silver.fact_laps``` | Clean lap times, convert to seconds, validate ranges | session_id (FK), driver_id (FK), lap_number, lap_time_sec, position, tire_compound |
| ```silver.fact_weather``` | Clean weather readings | session_id (FK), timestamp, air_temp, track_temp, humidity, wind_speed |
| ```silver.fact_results``` | Clean results, calculate positions | session_id (FK), driver_id (FK), position, points, status, grid_position |

**Gold Tables**<br>
| Table | Type | Description |
|---|---|---|
| ```gold.dim_drivers``` | Dimension | Driver info (with SCD Type 2 history) |
| ```gold.dim_constructors``` | Dimension | Constructor info |
| ```gold.dim_circuits``` | Dimension | Circuit info |
| ```gold.dim_sessions``` | Dimension | Session info with circuit and season |
| ```gold.fact_lap_times``` | Fact | Lap times at session/driver/lap grain |
| ```gold.fact_pit_stops``` | Fact | Pit stop events (from lap data) |
| ```gold.fact_race_results``` | Fact | Final results per driver per race |
| ```gold.agg_constructor_reliability``` | Aggregate | DNF rates, mechanical failure rates per constructor per season |

### Silver Layer ###
The ```Silver``` layer transforms ```Bronze``` data from *raw* into typed, clean, conformed, and trustworthy. ```Bronze``` has data in ```Delta tables``` as strings, but still as strings, with inconsistent column names, missing values, duplicate rows (from re-fetches) and no relationships between tables.

```Silver``` is where we enforce data quality, establish a canonical schema, deduplicate, and create the entity relationships (Primary Keys, Foreign Keys) that make the data queryable as a coherent database.

**Cleaning Transformations**<br>
- *Type Ccasting*: converts string columns to proper types;
- *```NULL``` Handling*: replace ```NULL```s (empty strings) with actual ```None``` or ```NaN```;
- *Deduplication*: although ```Bronze```'s ```MERGE``` should prevent duplicates, edge cases can still occur, thus the dedeuplication is based on the Primary Key, keeping the most recent fetch;
- *Column Standardization*: rename all columns to ```snake_case```, ensure naming convention;

**Conformation Transformations**<br>
The following actions aim at making tables work together:
- *Surrogate Keys*: each ```dimension table``` gets an integer surrogate key (```driver_id```, ```circuit_id```, etc) that's stable across schema changes; ```fact tables``` reference these surrogate keys;
*Foreign Key Relationships*: ```fact tables``` include FK columns that reference dimension tables;
- *Consistent Grain*: each ```fact table``` has a well-defined grain (one row per what?);

**Data Quality Gates**<br>
Between ```Bronze``` and ```Silver```, we apply data quality checks. If a check fails, we either fix the data (if the fix is deterministic) or quarantine the row (if it requires investigation):
- *Primary Key Uniqueness*: no duplicate rows by Primary Key;
- *Foreign Key Validity*: every FK in a ```fact table``` exists in the corresponding ```dimension table```;
- *Range Validation*: lap times between 30 and 600 seconds; speeds between 0 and 400 km/h; tire age >= 0;
- *Completeness*: critical columns have no ```NULL```s;
- *Referential Integrity*: every lap belongs to a valid session, every result belongs to a valid driver;

### Gold Layer ###
The ```Gold``` layer is the consumption layer (the layer the users interact with). It often includes pre-computed aggregate tables, which materialize common calculations so downstream consumers don't need to recompute them every time. These aggregates are computed by joining and grouping ```silver fact``` tables.

### Feature Store ###
A ```feature store``` is a system that *manages the lifecycle of ML features* (how they're defined, computed, stored, discovered, and served to training pipelines). It solves three critical problems in ML engineering:

**Feature Reuse**<br>
Without a ```feature store```, every user independently writes code to compute the same features. This leads to duplicated work, inconsistent definitions, and divergent results. A ```feature store``` *centralizes feature definitions so they're computed once and reused across teams*.

**Training-Serving Skew**<br>
In a typical workflow, a user computes features in a notebook (draft), trains a model, then hands the model to an engineer who must reimplement the same feature logic in a production pipeline. If the two implementations diverge, even slightly, the model receives different features at inference time than it saw during training (training-serving skew), and it's one of the most common causes of model degradation in production.

A ```feature store``` eliminates skew by *providing a single, versioned source-of-truth for feature values* (both training and inference read from the same feature definitions).

**Point-in-Time Correctness**<br>
When generating a training dataset, each training example represents an event at a specific point in time. The *features for that example must include only data that was available before the event*. If a feature accidentally includes data from after the event, the model learns from information it won't have at inference time (data leakage), and it produces models that look great in training but fail in production.

A ```feature store``` enforces point-in-time correctness through ```as-of joins```: for each training example with event timestamp ```T```, the ```feature store``` returns the most recent feature values that were computed before ```T```. 

### Project ###
**1) Virtual Environment and Dependencies**<br>
We create a virtual environment (```.venv```) and install the dependencies in the ```requirements.txt``` file. Dependencies include ```deltalake``` and ```duckdb``` (for core lakehouse), ```pandas``` and ```pyarrow``` (for data manipulation), ```fastf1``` (for data), and ```scikit-learn``` and ```mlflow``` (for ML).

**2) Lakehouse Config**<br>
We manually create the folder structure above in the root (```lakehouse``` folder), the ```config``` and the ```catalog``` (an empty ```JSON``` file; content should be ```{}```). The ```catalog``` class maps logical table names to physical paths and stores metadata (schema version, partition columns).

In [2]:
class LakehouseConfig: # Central config for lakehouse
    def __init__(self, root_path= "./lakehouse"):
        # Root
        self.root = root_path
        # Tiers
        self.raw = os.path.join(self.root, "raw")
        self.bronze = os.path.join(self.root, "bronze")
        self.silver = os.path.join(self.root, "silver")
        self.gold = os.path.join(self.root, "gold")
        self.feature_store = os.path.join(self.root, "feature_store")
        self.cache = os.path.join(self.root, "cache")
        # Catalog
        self.catalog_path = os.path.join(self.root, "catalog.json")

    def tier_path(self, tier): # Returns path for given tier
        tiers = {"raw": self.raw,
                 "bronze": self.bronze,
                 "silver": self.silver,
                 "gold": self.gold,
                 "feature_store": self.feature_store}
        if tier not in tiers:
            raise ValueError(f"Unknown tier: {tier}. Valid zones: {list(tiers.keys())}")
        return tiers[tier]

    def table_path(self, tier, table_name): # Returns path for table in tier
        return os.path.join(self.tier_path(tier), table_name)

In [3]:
class Catalog:
    def __init__(self, config):
        # Config
        self.config = config
        # Catalog 
        self.entries = {} 
        self.load_catalog() 

    def load_catalog(self): # Load existing catalog from disk or start fresh
        # Creating the catalog path
        if os.path.isfile(self.config.catalog_path):
            with open(self.config.catalog_path, "r") as _file:
                self.entries = json.load(_file)
        else:
            pass

    def register(self, tier, table_name, description, partition_cols= []): # Register a table in the catalog and returns table path
                                                                           #   tier: 'bronze', 'silver', 'gold', 'feature_store'
                                                                           #   table_name: human-readable name
                                                                           #   partition_cols: list of cols names for partitioning
                                                                           #   description: human-readable description
        # Build the logical key as "tier.table_name"
        key = f"{tier}.{table_name}"
        path = self.config.table_path(tier, table_name)
        # Update catalog entry
        self.entries[key] = {"path": path,
                             "tier": tier,
                             "description": description,
                             "partition_cols": partition_cols,
                             "created_at": datetime.now().isoformat(), # UTC
                             "schema_version": 1}
        # Persist the catalog to disk
        self.save()
        return path

    def get_path(self, table_name): # Returns the physical path for a table ('zone.table_name' format)
        if table_name not in self.entries:
            raise KeyError(f"Table '{table_name}' not found in catalog."
                           f"Available: {list(self.entries.keys())}")
        return self.entries[table_name]["path"]

    def get_metadata(self, table_name): # Returns the full metadata dict for a table
        if table_name not in self.entries:
            raise KeyError(f"Table '{table_name}' not found in catalog.")
        return self.entries[table_name]

    def list_tables(self, tier= None): # List all tables (optionally filtered by tier)
        if tier:
            return {key: entry for key, entry in self.entries.items() if entry["tier"] == tier}
        return self.entries

    def increment_schema_version(self, table_name): # Increments schema version for table (used for evolution)
        if table_name not in self.entries:
            raise KeyError(f"Table '{table_name}' not found in catalog.")
        # Increment version and persist
        self.entries[table_name]["schema_version"] += 1
        self.save()
        return self.entries[table_name]["schema_version"]

    def save(self): # Persists the catalog to disk
        with open(self.config.catalog_path, "w") as _file:
            json.dump(self.entries, _file, indent= 2) # 'indent': indents nesting level by number of spaces

**3) Delta Table Utilities**<br>
Another class to wrap ```deltalake``` API functionalities and provide a consistent interface across all pipeline stages.

In [4]:
class DeltaLakeWrapper:
    def __init__(self):
        pass

    def write_table(self, df, 
                    path, # 'path': destination path for Delta table (str)
                    mode= "overwrite", # 'mode': write mode ('overwrite', 'append', 'ignore')
                    partition_cols= None, # 'partition_cols': list of partition column names
                    schema_mode= None): # 'schema_mode': schema behavior ('overwrite' to replace; 'merge' to evolve)
        ## Writes DataFrame to Delta table
        # Build kwargs for write
        kwargs = {"mode": mode,
                  "partition_by": partition_cols}
        # Add 'schema_mode' if provided 
        if schema_mode:
            kwargs["schema_mode"] = schema_mode
        # Writes to Delta Lake
        write_deltalake(path, df, **kwargs) # 'write_deltalake': handles conversion from Pandas to Arrow internally
        return DeltaTable(path) # 'DeltaTable(path)': loads Delta table by path

    def read_table(self, path, version= None): # Reads Delta table as DataFrame
        # Loads Delta table by path
        dt = DeltaTable(path)
        # Time travel to specific version if provided
        if version is not None:
            dt.load_version(version)
        return dt.to_pandas() # Converts to DataFrame

    def table_exists(self, path): # Checks if table exists for path
        # Delta table is identified by presence of '_delta_log' directory
        return os.path.exists(os.path.join(path, "_delta_log"))

    def get_table_version(self, path): # Returns current version for Delta table
        return DeltaTable(path).version()

    def merge_table(self, path,
                    new_df, # 'new_df': new data to merge (DataFrame)
                    merge_condition, # 'merge_condition': SQL condition for rows matching (a ON statement)
                    when_matched_update= True, # 'when_matched_update': if True, update matched rows
                    when_not_matched_insert= True): # 'when_not_matched_insert': if True, insert unmatched rows
        ## Performs Delta MERGE (upsert) operation(core idempotency mechanism for bronze layer: re-running the pipeline updates records rather than duplicating them)
        # Loads Delta table by path
        dt = DeltaTable(path)
        # Build the merge builder
        builder = dt.merge(source= new_df, # 'source': source DataFrame
                           predicate= merge_condition, # 'predicate': SQL condition to match source rows with Delta table rows
                           source_alias= "source", # 'source_alias': alias for source to be used in 'predicate'
                           target_alias= "target") # 'target_alias': alias for Delta table to be used in 'predicate'
        # Configure matched/not-matched actions if provided
        if when_matched_update:
            builder = builder.when_matched_update_all() # Updates all rows when 'predicate' matches
        if when_not_matched_insert:
            builder = builder.when_not_matched_insert_all() # Inserts all rows when 'predicate' does not match
        # Commit and apply changes defined by builder
        result = builder.execute()
        return result # Returns dict with merge summary

    def delete_table(self, path, # 'path': Delta table path (str)
                     retention_hours= 24*7, # 'retention_hours': minimum age of files to remove (default: 7 days)
                     dry_run= True): # 'dry_run': if True, list files that would be removed without deleting
        ## Recursively deletes old unreferenced files from Delta table (default retention: 7 days)
        # Loads Delta table by path
        dt = DeltaTable(path)
        return dt.vacuum(retention_hours= retention_hours, 
                         dry_run= dry_run, 
                         enforce_retention_duration= False) # 'enforce_retention_duration': if True, prevents from deleting files newer than 'retention_hours'

    def optimize_table(self, path, # 'path': Delta table path (str)
                       z_order_cols= None): # 'z_order_cols': columns to Z-order (list)
        ## Compacts small files in Delta table (optionally Z-order by columns - allocates rows with similar values for specified columns together to improve query performance) 
        # Loads Delta table by path
        dt = DeltaTable(path)
        return dt.optimize.z_order(z_order_cols) if z_order_cols else dt.optimize.compact() # 'optimize.compact()': performs file compaction (bin-packing)

**4) Ingesting ```FastF1``` Data**<br>
We'll fetche F1 session data (laps, weather, results, session metadata) using the ```FastF1``` library as source and write it to ```Bronze``` tables.

In [5]:
class Ingestor:
    def __init__(self, catalog, dl_wrapper):
        # Config
        self.catalog = catalog
        # Wrapper
        self.dl_wrapper = dl_wrapper
        # Constants
        self.session_types = ["FP1", "FP2", "FP3", "Q", "R"] # Session types we want to fetch from events
                                                             # Available: 'R': race, 'Q': Qualifying, 'FP1'/'FP2'/'FP3': Free Practice
        self.session_type_names = {"FP1": "Free Practice 1", "FP2": "Free Practice 2", "FP3": "Free Practice 3", "Q": "Qualifying", "R": "Race"}
        self.laps_map = {"Time": "lap_time", # Columns to extract from FastF1 DataFrame, mapped to Bronze schema
                         "Driver": "driver_code",
                         "DriverNumber": "driver_number",
                         "LapTime": "lap_time_duration",
                         "LapNumber": "lap_number",
                         "Stint": "stint_number",
                         "PitOutTime": "pit_out_time",
                         "PitInTime": "pit_in_time",
                         "Sector1Time": "sector_1_time",
                         "Sector2Time": "sector_2_time",
                         "Sector3Time": "sector_3_time",
                         "Sector1SessionTime": "sector_1_session_time",
                         "Sector2SessionTime": "sector_2_session_time",
                         "Sector3SessionTime": "sector_3_session_time",
                         "SpeedI1": "speed_i1",
                         "SpeedI2": "speed_i2",
                         "SpeedFL": "speed_fl",
                         "SpeedST": "speed_st",
                         "IsPersonalBest": "is_personal_best",
                         "Compound": "tire_compound",
                         "TyreLife": "tire_age",
                         "FreshTyre": "is_fresh_tire",
                         "Team": "team",
                         "LapStartTime": "lap_start_time",
                         "LapStartDate": "lap_start_date",
                         "TrackStatus": "track_status",
                         "Position": "position",
                         "Deleted": "is_deleted",
                         "DeletedReason": "deleted_reason",
                         "FastF1Generated": "is_fastf1_generated",
                         "IsAccurate": "is_accurate"}
        self.weather_map = {"Time": "weather_time",
                            "AirTemp": "air_temp",
                            "Humidity": "humidity",
                            "Pressure": "pressure",
                            "Rainfall": "is_rain",
                            "TrackTemp": "track_temp",
                            "WindDirection": "wind_direction",
                            "WindSpeed": "wind_speed"}
        self.results_map = {"DriverNumber": "driver_number",
                            "BroadcastName": "broadcast_name",
                            "Abbreviation": "driver_code",
                            "DriverId": "driver_id",
                            "TeamName": "team",
                            "TeamId": "team_id",
                            "Position": "position",
                            "ClassifiedPosition": "classified_position",
                            "GridPosition": "grid_position",
                            "Q1": "q1_time",
                            "Q2": "q2_time",
                            "Q3": "q3_time",
                            "Time": "total_time",
                            "Status": "status",
                            "Points": "points"}

    ## Helper Methods
    def add_metadata(self, df, batch_id): # Add metadata cols to Bronze DataFrame (track when data was fetched, source, and pipeline that ran it)
        # Deep copy 'df'
        df = df.copy()
        # Add metadata
        df["_fetch_ts"] = datetime.now(timezone.utc).isoformat() # UTC
        df["_source"] = "fastf1"
        df["_batch_id"] = batch_id
        return df

    ## Extract Methods
    # 'session': object represents a specific stage of a race weekend (event)
    def extract_session(self, session, season, round_num, session_type): # Extracts session's event data 
        # Access session's event (parent)
        event = session.event # Available attr: 'RoundNumber', 'Country', 'Location', 'EventName', 'OfficialEventName', 
                                #                 'EventDate', 'EventFormat', 'Session[X]', 'Session[X]Date', 'Session[X]DateUtc', 
                                #                 'F1ApiSupport'
        # Pull metada (as lists for DataFrame conversion)
        data = {"season": [season],
                "round": [round_num],
                "session_type": [session_type],
                "session_name": [self.session_type_names.get(session_type, session_type)], # Defaults to 'session_type' if not found
                "event_name": [event.get("EventName", "")],
                "event_date": [str(event.get("EventDate", ""))],
                "country": [event.get("Country", "")],
                "location": [event.get("Location", "")],
                "circuit_name": [event.get("OfficialEventName", "")],
                "session_date": [str(session.date) if hasattr(session, "date") else ""]}
        return pd.DataFrame(data).astype(str) # Convert to string DataFrame (single-row)

    def extract_laps(self, session, season, round_num, session_type): # Extracts 'laps' data from session and flattens to schema
                                                                # Each 'laps' contain several 'lap' (1 row each)
        # Access session's laps from all drivers ('session.load()' must have 'laps=True')
        laps = session.laps # Available attr: 'Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime',
                            #                 'Sector[X]Time', 'Sector[X]SessionTime', 'SpeedI[X]', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
                            #                 'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime', 'LapStartDate', 'TrackStatus',
                            #                 'Position', 'Deleted', 'DeletedReason', 'FastF1Generated', 'IsAccurate'
        if laps is None or len(laps) == 0: # If empty...
            return pd.DataFrame()
        
        # Loop and rename columns
        available = {col: lap for col, lap in self.laps_map.items() if col in laps.columns}
        df = laps[list(available.keys())].copy() # Deep copy
        df.columns = [available[col] for col in df.columns]
        # Add context columns (which session laps belong to)
        df["season"] = season
        df["round"] = round_num
        df["session_type"] = session_type
        return df.astype(str)

    def extract_weather(self, session, season, round_num, session_type): # Extracts weather data from session
        try:
            # Access session's weather ('session.load()' must have 'weather=True')
            weather = session.weather_data # Available attr: 'Time', 'AirTemp', 'Humidity', 'Pressure', 'Rainfall', 'TrackTemp', 'WindDirection', 'WindSpeed'
            if weather is None or len(weather) == 0: # If empty...
                return pd.DataFrame()
        except Exception as e:
            print(f"[Warning] No weather data: {e}")
            return pd.DataFrame()

        # Select and rename columns
        available = {col: col_name for col, col_name in self.weather_map.items() if col in weather.columns}
        df = weather[list(available.keys())].copy() # Deep copy
        df.columns = [available[col] for col in df.columns]
        df["season"] = season
        df["round"] = round_num
        df["session_type"] = session_type
        return df.astype(str)

    def extract_results(self, session, season, round_num, session_type): # Extracts session's results (final classification) 
        try:
            # Access session's results
            results = session.results # Available attr: 'DriverNumber', 'BroadcastName', 'FullName', 'Abbreviation', 'DriverId', 'TeamName', 'TeamColor',
                                        #                 'TeamId', 'FirstName', 'LastName', 'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition', 
                                        #                 'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points', 'Laps'
            if results is None or len(results) == 0: # If empty...
                return pd.DataFrame()
        except Exception as e:
            print(f"[Warning] No results data: {e}")
            return pd.DataFrame()

        # Select and rename columns
        available = {col: col_name for col, col_name in self.results_map.items() if col in results.columns}
        df = results[list(available.keys())].copy() # Deep copy
        df.columns = [available[col] for col in df.columns]
        df["season"] = season
        df["round"] = round_num
        df["session_type"] = session_type
        return df.astype(str)

    ## Bronze Write Methods
    def write_bronze(self, df, table_name, merge_condition= None, partition_cols= None): # Writes DataFrame to bronze Delta table
        if df is None or len(df) == 0: # If empty...
            print(f"[Info] No data for bronze.{table_name}, skipping.")
            return 0
        # Get table path
        path = self.catalog.config.table_path("bronze", table_name)
        # If table exists and 'merge_condition' is provided, uses merge for idempotency...
        if self.dl_wrapper.table_exists(path) and merge_condition:
            # Idempotent Upsert: match existing rows by key (Update or Insert) 
            self.dl_wrapper.merge_table(path, df, merge_condition)
            print(f"[Info] Merged {len(df)} rows into 'bronze.{table_name}'")
        else: 
            # ...else creates the table ('overwrite' on first run)
            self.dl_wrapper.write_table(df, path, mode= "overwrite", partition_cols= partition_cols or ["season"]) # 'partition_cols': if None/False, defaults to '["season"]'
            # Register in catalog
            catalog.register("bronze", table_name,
                                partition_cols= partition_cols or ["season"],
                                description= f"Bronze layer: {table_name} ")
            print(f"[Info] Created bronze.{table_name} with {len(df)} rows")
        return len(df)

    ## Ingestion Methods
    def ingest_season(self, season): # Ingest all sessions (from all rounds) for given season into bronze layer
        # Enable cache to use cache tier
        fastf1.Cache.enable_cache(cache_dir= self.catalog.config.cache) # 'cache_dir': directory path to store cached data
        # Generate a 'batch_id' for run
        batch_id = str(uuid.uuid4())
        print(f"[Info] Ingestion for season {season} (batch: {batch_id})")
        # Get event schedule for season (includes testing)
        schedule = fastf1.get_event_schedule(season) # Available attr: 'RoundNumber', 'Country', 'Location', 'OfficialEventName', 'EventName', 'EventDate', 'EventFormat', 'Session[X]', 'Session[X]Date', 'Session[X]DateUtc', 'F1ApiSupport'    
        # Skip testing and non-race events (RoundNumber == 0 is testing)
        schedule = schedule[schedule["RoundNumber"] > 0]
        # Process stats dict initialize
        stats = {"sessions_processed": 0,
                    "sessions_failed": 0,
                    "rows_by_table": {}}
        # Iterate over each event
        for idx, event in schedule.iterrows():
            round_num = event["RoundNumber"]
            event_name = event.get("EventName", f"Round {round_num}")
            print(f"[Info] Processing: Season {season} - {event_name} (Round {round_num})")

            # Iterate over each mapped session type
            for session_type in self.session_types:
                try:
                    # Load the session
                    session = fastf1.get_session(season, round_num, session_type)
                    session.load()  # Fetch data
                    # Increment stats
                    stats["sessions_processed"] += 1
                    print(f"[Info] Loaded {session_type} session")
                    # Extract each dataset
                    session_df = self.extract_session(session, season, round_num, session_type)
                    laps_df = self.extract_laps(session, season, round_num, session_type)
                    weather_df = self.extract_weather(session, season, round_num, session_type)
                    results_df = self.extract_results(session, season, round_num, session_type)

                    # Add metadata columns to DataFrames
                    for df in [session_df, laps_df, weather_df, results_df]:
                        if len(df) > 0: # If not empty...
                            df = self.add_metadata(df, batch_id)

                    ## Write to bronze with idempotent MERGE (each table has unique merge condition)
                    # Sessions
                    row_count = self.write_bronze(df= session_df, 
                                                  table_name= "sessions",
                                                  merge_condition= "target.season = source.season AND target.round = source.round AND target.session_type = source.session_type")
                    stats["rows_by_table"]["sessions"] = stats["rows_by_table"].get("sessions", 0) + row_count
                    # Laps
                    row_count = self.write_bronze(df= laps_df, 
                                                  table_name= "laps",
                                                  merge_condition= "target.season = source.season AND target.round = source.round AND target.session_type = source.session_type AND target.driver_number = source.driver_number AND target.lap_number = source.lap_number")
                    stats["rows_by_table"]["laps"] = stats["rows_by_table"].get("laps", 0) + row_count
                    # Weather
                    row_count = self.write_bronze(df= weather_df, 
                                                  table_name= "weather",
                                                  merge_condition= "target.season = source.season AND target.round = source.round AND target.session_type = source.session_type AND target.weather_time = source.weather_time")
                    stats["rows_by_table"]["weather"] = stats["rows_by_table"].get("weather", 0) + row_count
                    # Results
                    row_count = self.write_bronze(df= results_df, 
                                                  table_name= "results",
                                                  merge_condition="target.season = source.season AND target.round = source.round AND target.session_type = source.session_type AND target.driver_number = source.driver_number")
                    stats["rows_by_table"]["results"] = stats["rows_by_table"].get("results", 0) + row_count
                except Exception as e:
                    stats["sessions_failed"] += 1
                    print(f"[Error] Failed to load {session_type} for round {round_num}: {e}")
        print(f"[Info] Ingestion complete for season {season}")
        return stats

In [6]:
# Instantiating classes
config = LakehouseConfig()
catalog = Catalog(config) # Used in dependency injections
dl_wrapper = DeltaLakeWrapper()
ingestor = Ingestor(catalog, dl_wrapper)

In [ ]:
# Ingesting the 2025 season
ingestor.ingest_season(2025)

**5) Verifying the Bronze Layer**<br>
After running the pipeline, we verify the bronze layer (list all tables and check row counts / sample data):

In [81]:
# List all bronze tables
print("Bronze Tables:")
for name, info in catalog.list_tables("bronze").items():
    print(f"\t{name}: {info['path']}")

Bronze Tables:
	bronze.sessions: ./lakehouse\bronze\sessions
	bronze.laps: ./lakehouse\bronze\laps
	bronze.weather: ./lakehouse\bronze\weather
	bronze.results: ./lakehouse\bronze\results


In [82]:
# Check row counts and sample data
for table in ["sessions", "laps", "results", "weather"]:
    try:
        path = catalog.get_path(f"bronze.{table}")
        df = dl_wrapper.read_table(path)
        print(f"bronze.{table}: {len(df)} rows, version {dl_wrapper.get_table_version(path)}")
    except KeyError:
        print(f"bronze.{table}: not registered") # Table not registered
    except Exception as e:
        print(f"bronze.{table}: error ({e})") # Error

bronze.sessions: 108 rows, version 306
bronze.laps: 62131 rows, version 304
bronze.results: 2159 rows, version 298
bronze.weather: 11094 rows, version 298


**6) Confirm Schemas**<br>
Now that we have the data ingested, we can confirm the ```Bronze``` tables schemas and (desired) data types (query below):
- ```bronze.sessions```
    - ```season```: ```int```  	
    - ```round```: ```int``` 	
    - ```session_type```: ```str``` 
    - ```session_name```: ```str``` 
    - ```event_name```: ```str``` 
    - ```event_date```: ```date``` 
    - ```country```: ```str``` 
    - ```location```: ```str``` 
    - ```circuit_name```: ```str``` 
    - ```session_date```: ```date```

- ```bronze.laps```
    - ```lap_time```: ```int``` (seconds)	
    - ```driver_code```: ```str```	
    - ```driver_number```: ```int```	
    - ```lap_time_duration```: ```int``` (seconds)		
    - ```lap_number```: ```int```
    - ```stint_number```: ```int```
    - ```pit_out_time```: ```int``` (seconds)	
    - ```pit_in_time```: ```int``` (seconds)	
    - ```sector_1_time```: ```int``` (seconds)	
    - ```sector_2_time```: ```int``` (seconds)		
    - ```sector_3_time```: ```int``` (seconds)	
    - ```sector_1_session_time```: ```int``` (seconds)	
    - ```sector_2_session_time```: ```int``` (seconds)	
    - ```sector_3_session_time```: ```int``` (seconds)	
    - ```speed_i1```: ```float```
    - ```speed_i2```: ```float```
    - ```speed_fl```: ```float```
    - ```speed_st```: ```float```
    - ```is_personal_best```: ```bool```
    - ```tire_compound```: ```str```
    - ```tire_age```: ```int```
    - ```is_fresh_tire```: ```bool```
    - ```team```: ```str```
    - ```lap_start_time```: ```int``` (seconds; relative to start of session)
    - ```lap_start_date```: ```timestamp```
    - ```track_status```: ```int``` (1: green flag, 2: yellow flag, 3: None, 4: safety car, 5: red flag, 6: virtual safety car deployed, 7: virtual safety car ended - ‘1’ marks the actual end)
    - ```position```: ```int```
    - ```is_deleted```: ```bool```
    - ```deleted_reason```: ```str```
    - ```is_fastf1_generated```: ```bool```
    - ```is_accurate```: ```bool```
    - ```season```: ```int```
    - ```round```: ```int```
    - ```session_type```: ```str```

- ```bronze.results```
    - ```driver_number```: ```int```
    - ```broadcast_name```: ```str```
    - ```driver_code```: ```str```
    - ```driver_id```: ```str```
    - ```team```: ```str```
    - ```team_id```: ```str```
    - ```position```: ```int```
    - ```classified_position```: ```int```
    - ```grid_position```: ```int```
    - ```q1_time```: ```int``` (seconds)
    - ```q2_time```: ```int``` (seconds)
    - ```q3_time```: ```int``` (seconds)
    - ```total_time```: ```int``` (seconds)
    - ```status```: ```str```
    - ```points```: ```int```
    - ```season```: ```int```
    - ```round```: ```int```
    - ```session_type```: ```str```
    - ```__index_level_0__```: ```int```

- ```bronze.weather```
    - ```weather_time```: ```int``` (seconds)
    - ```air_temp```: ```float```
    - ```humidity```: ```float```
    - ```pressure```: ```float```
    - ```is_rain```: ```bool```
    - ```track_temp```: ```float```
    - ```wind_direction```: ```int```
    - ```wind_speed```: ```float```
    - ```season```: ```int```
    - ```round```: ```int```
    - ```session_type```: ```str```

In [10]:
# Querying ('sessions', 'laps', 'results', 'weather')
dt = DeltaTable(catalog.get_path(f"bronze.weather"))
dt.to_pandas().head(3)

,weather_time,air_temp,humidity,pressure,is_rain,track_temp,wind_direction,wind_speed,season,round,session_type
0,0 days 00:01:00.108000,27.4,55.0,1016.4,False,34.6,67,3.0,2025,24,R
1,0 days 00:02:00.124000,27.4,54.0,1016.4,False,34.4,290,2.4,2025,24,R
2,0 days 00:03:00.131000,27.5,54.0,1016.3,False,34.4,286,1.4,2025,24,R


**7) Bronze to Silver Transformations**<br>
We read ```raw``` data from ```Bronze```, apply cleansing and conformation rules, and write the result to ```Silver```. Each ```build_``` method:
1. Reads ```Bronze``` table(s) into ```DataFrame```(s);
2. Applies transformations;
3. Runs data quality checks;
4. Write to ```Silver```;

In [7]:
class SilverUtils:
    def __init__(self):
        pass

    # Quality Methods
    def assert_unique(self, df, cols, table_name): # Asserts DataFrame has no duplicate rows by cols
        dupes = df.duplicated(subset= cols, keep= False) # 'duplicated()': returns bool Seriesfor duplicate rows
                                                         #    'subset': specific columns
                                                         #    'keep': which duplicates to mark as True (Available: 'first', 'last', False) 
        if dupes.any(): # If not empty...
            raise AssertionError(f"[Quality] Unique Fail: {table_name} has {dupes.sum()} duplicates")
        print(f"[Quality] Unique Pass: {table_name} unique by {cols}")

    def assert_no_nulls(self, df, cols, table_name): # Asserts cols have no NULL values
        for col in cols:
            null_count = df[col].isna().sum()
            if null_count > 0:
                raise AssertionError(f"[Quality] NULLs Fail: {table_name}.{col} has {null_count} NULLs")
        print(f"[Quality] NULLs Pass: {table_name} has no NULLs in {cols}")

    def assert_range(self, df, col, min_val, max_val, table_name): # Asserts numeric cols' values fall within [min_val, max_val] range
        non_null = df[col].dropna()
        if len(non_null) == 0: # If no non-null values...
            return
        violations = ((non_null < min_val) | (non_null > max_val)).sum() # Count of values outside range
        if violations > 0:
            raise AssertionError(f"[Quality] Range Fail: {table_name}.{col} has {violations} values outside [{min_val}, {max_val}]")
        print(f"[Quality] Range Pass: {table_name}.{col}")

    # Conversion Methods
    def tdstr_to_int(self, value): # Converts 'value' to int    
        try:
            if isinstance(value, str):
                return pd.to_timedelta(value).total_seconds() if not pd.isna(value) else None
        except (ValueError, TypeError):
            return None
            
    def str_to_int(self, value): # Converts 'value' to int    
        try:
            if isinstance(value, str):
                return int(float(value)) if not pd.isna(value) else None
        except (ValueError, TypeError):
            return None

    def str_to_float(self, value): # Converts 'value' to float
        try:
            if isinstance(value, str):
                return float(value) if not pd.isna(value) else None
        except (ValueError, TypeError):
            return None

In [8]:
class SilverBuider:
    def __init__(self, catalog, dl_wrapper, silver_utils):
        # Config
        self.catalog = catalog
        # Wrapper
        self.dl_wrapper = dl_wrapper
        # Utils
        self.silver_utils = silver_utils
        
    ## Dimension Tables Builders 
    def build_dim_drivers(self, bronze_results_df, bronze_laps_df): # Builds 'dim_drivers' table from Bronze's 'result' and 'laps' tables
                                                                    # Extracts unique drivers across all sessions, assigns surrogate keys, and implements SCD Type 2 for team changes across seasons
        print("Building 'silver.dim_drivers'...")
        # Collect unique driver info from 'results'
        driver_cols = ["driver_number", "driver_code", "broadcast_name", "driver_id", "team", "season"]
        available_results = [col for col in driver_cols if col in bronze_results_df.columns]
        drivers_results = bronze_results_df[available_results].copy()
        # Collect any drivers from 'laps' not in 'results'
        if "driver_code" in bronze_laps_df.columns and "team" in bronze_laps_df.columns:
            lap_driver_cols = ["driver_code", "driver_number", "team", "season"]
            available_laps = [col for col in lap_driver_cols if col in bronze_laps_df.columns]
            drivers_laps = bronze_laps_df[available_laps].copy()
            drivers_laps["broadcast_name"] = None
            drivers_laps["driver_id"] = None
            # Combine both sources
            all_drivers = pd.concat([drivers_results, drivers_laps], ignore_index= True)
        else:
            all_drivers = drivers_results

        ## Creating columns
        # Normalize names: use 'broadcast_name' (fallback to 'driver_code')
        all_drivers["full_name"] = all_drivers["broadcast_name"].fillna(all_drivers["driver_code"])
        # Sort by season to detect team changes chronologically
        all_drivers = all_drivers.sort_values(["driver_number", "season"])

        # Detect team changes per driver across seasons (SCD Type 2; each driver's team is compared to previous season's team)
        all_drivers["prev_team"] = all_drivers.groupby("driver_number")["team"].shift(1) # 'shift(periods)': if periods > 0, then pushes rows down (NaN for vallues not found)
        all_drivers["team_changed"] = all_drivers["team"] != all_drivers["prev_team"] # Bool flag for team change
        # First occurrence per driver is also a "change" (new record)
        all_drivers["team_changed"] = all_drivers["team_changed"].fillna(True)
        # Keep only rows where team changed (SCD Type 2: one record per team change)
        scd_records = all_drivers[all_drivers["team_changed"]].copy()
        # Deduplicate (by driver_number + team, keep first season for each team)
        scd_records = scd_records.drop_duplicates(subset= ["driver_number", "team"], keep= "first")

        # Assign surrogate key (incremental integer)
        scd_records = scd_records.sort_values(["driver_number", "season"])
        scd_records["driver_sk"] = range(1, len(scd_records) + 1) 
        # SCD Type 2 columns
        scd_records["valid_from"] = scd_records["season"].astype(str) + "-01-01" # Date formatted
        scd_records["valid_to"] = None  # Will be filled by next record's 'valid_from' (below)
        scd_records["is_current"] = True

        # Set 'valid_to' for previous records of the same driver
        for driver_num in scd_records["driver_number"].unique():
            driver_records = scd_records[scd_records["driver_number"] == driver_num].sort_values("season") # Sorts current filtered driver rows per season
            if len(driver_records) > 1:
                for i in range(len(driver_records) - 1): # Looping...
                    idx = driver_records.index[i]
                    next_idx = driver_records.index[i + 1]
                    next_season = scd_records.loc[next_idx, "season"]
                    scd_records.loc[idx, "valid_to"] = str(int(next_season)) + "-01-01" # Updates 'valid_to'
                    scd_records.loc[idx, "is_current"] = False

        # Select final columns
        final_df = scd_records[["driver_sk", "driver_number", "driver_code", "full_name", 
                              "team", "season", "valid_from", "valid_to", "is_current" ]].rename(columns={"driver_sk": "driver_id", "season": "first_season_seen"})
        # Data quality checks
        self.silver_utils.assert_unique(final_df, ["driver_id"], "silver.dim_drivers")
        self.silver_utils.assert_no_nulls(final_df, ["driver_id", "driver_number"], "silver.dim_drivers")
        return final_df

    def build_dim_constructors(self, bronze_results_df): # Builds 'dim_constructors' table from Bronze's 'result' table
        print("Building 'silver.dim_constructors'...")
        # If 'team' col is not present, returns empty table
        if "team" not in bronze_results_df.columns:
            return pd.DataFrame()

        # Extract unique teams
        teams = bronze_results_df[["team"]].drop_duplicates().dropna() # Drops duplicates and NAs
        teams = teams.sort_values("team").reset_index(drop= True) # Sorts
        teams["constructor_id"] = range(1, len(teams) + 1) # Assigns surrogate key (incremental integer)
        teams["constructor_name"] = teams["team"] # Assigns value
        final_df = teams[["constructor_id", "constructor_name"]]

        # Data quallity checks
        self.silver_utils.assert_unique(final_df, ["constructor_id"], "silver.dim_constructors")
        return final_df

    def build_dim_circuits(self, bronze_sessions_df): # Builds 'dim_circuits' table from Bronze's 'sessions' table
        print("Building 'silver.dim_circuits'...")
        # If 'circuit_name' col is not present, returns empty table
        if "circuit_name" not in bronze_sessions_df.columns:
            return pd.DataFrame()

        # Extract unique circuits
        circuits = bronze_sessions_df[["circuit_name", "country", "location"]].drop_duplicates() # Drops duplicates
        circuits = circuits.dropna(subset= ["circuit_name"]) # Drops NAs
        circuits = circuits.sort_values("circuit_name").reset_index(drop=True) # Sorts
        circuits["circuit_id"] = range(1, len(circuits) + 1) # Assigns surrogate key (incremental integer)
        final_df = circuits[["circuit_id", "circuit_name", "country", "location"]]

        # Data quallity checks
        self.silver_utils.assert_unique(final_df, ["circuit_id"], "silver.dim_circuits")
        return final_df

    def build_dim_sessions(self, bronze_sessions_df, dim_circuits): # Builds 'dim_sessions' table from Bronze's 'sessions' metadata (links each session to its circuit via 'circuit_id')
        print("Building 'silver.dim_sessions'...")
        # Assign surrogate key (incremental integer)
        sessions = bronze_sessions_df.copy().sort_values(["season", "round", "session_type"])
        sessions["session_id"] = range(1, len(sessions) + 1)

        # Joins 'sessions' with 'dim_circuits' to get 'circuit_id' (FK)
        if len(dim_circuits) > 0: # If not empty...
            sessions = sessions.merge(dim_circuits[["circuit_id", "circuit_name"]],
                                      on= "circuit_name",
                                      how= "left")
        else:
            sessions["circuit_id"] = None
        final_df = sessions[["session_id", "season", "round", "session_type", "session_name",
                             "event_name", "country", "location", "circuit_name", "circuit_id",
                             "session_date", "event_date"]]

        # Data quallity checks
        self.silver_utils.assert_unique(final_df, ["session_id"], "silver.dim_sessions")
        self.silver_utils.assert_no_nulls(final_df, ["session_id", "season", "round", "session_type"], "silver.dim_sessions")
        return final_df

    ## Fact Tables Builders 
    def build_fact_laps(self, bronze_laps_df, dim_sessions, dim_drivers): # Builds 'fact_laps' table from Bronze's 'laps' table
        print("Building 'silver.fact_laps'...")
        # If 'bronze_laps_df' is empty...
        if len(bronze_laps_df) == 0:
            return pd.DataFrame()

        # Deep copy 'bronze_laps_df'
        df = bronze_laps_df.copy()

        # Convert time columns to float
        time_cols = ["lap_time_duration", "sector_1_time", "sector_2_time", "sector_3_time", "pit_out_time", "pit_in_time"]
        for col in time_cols: # Looping...
            if col in df.columns: # If col is present...
                df[col] = df[col].apply(self.silver_utils.str_to_float) # Float
        # Convert numeric columns to int/float
        if "season" in df.columns:
            df["season"] = df["season"].apply(self.silver_utils.str_to_int) # Int
        if "round" in df.columns:
            df["round"] = df["round"].apply(self.silver_utils.str_to_int) # Int
        if "stint_number" in df.columns:
            df["stint_number"] = df["stint_number"].apply(self.silver_utils.str_to_int) # Int
        if "position" in df.columns:
            df["position"] = df["position"].apply(self.silver_utils.str_to_int) # Int
        if "tire_age" in df.columns:
            df["tire_age"] = df["tire_age"].apply(self.silver_utils.str_to_float) # Float
        if "speed_i1" in df.columns:
            df["speed_i1"] = df["speed_i1"].apply(self.silver_utils.str_to_float) # Float
        if "speed_i2" in df.columns:
            df["speed_i2"] = df["speed_i2"].apply(self.silver_utils.str_to_float) # Float
        if "speed_fl" in df.columns:
            df["speed_fl"] = df["speed_fl"].apply(self.silver_utils.str_to_float) # Float
        if "speed_st" in df.columns:
            df["speed_st"] = df["speed_st"].apply(self.silver_utils.str_to_float) # Float
        if "lap_time" in df.columns:
            df["lap_time"] = df["lap_time"].apply(self.silver_utils.tdstr_to_int) # Int
        if "lap_time_duration" in df.columns:
            df["lap_time_duration"] = df["lap_time_duration"].apply(self.silver_utils.tdstr_to_int) # Int
        if "sector_1_time" in df.columns:
            df["sector_1_time"] = df["sector_1_time"].apply(self.silver_utils.tdstr_to_int) # Int
        if "sector_2_time" in df.columns:
            df["sector_2_time"] = df["sector_2_time"].apply(self.silver_utils.tdstr_to_int) # Int
        if "sector_3_time" in df.columns:
            df["sector_3_time"] = df["sector_3_time"].apply(self.silver_utils.tdstr_to_int) # Int
        if "sector_1_session_time" in df.columns:
            df["sector_1_session_time"] = df["sector_1_session_time"].apply(self.silver_utils.tdstr_to_int) # Int
        if "sector_2_session_time" in df.columns:
            df["sector_2_session_time"] = df["sector_2_session_time"].apply(self.silver_utils.tdstr_to_int) # Int
        if "sector_3_session_time" in df.columns:
            df["sector_3_session_time"] = df["sector_3_session_time"].apply(self.silver_utils.tdstr_to_int) # Int
        if "lap_start_time" in df.columns:
            df["lap_start_time"] = df["lap_start_time"].apply(self.silver_utils.tdstr_to_int) # Int
        if "pit_in_time" in df.columns:
            df["pit_in_time"] = df["pit_in_time"].apply(self.silver_utils.tdstr_to_int) # Int
        if "pit_out_time" in df.columns:
            df["pit_out_time"] = df["pit_out_time"].apply(self.silver_utils.tdstr_to_int) # Int

        # Join with 'dim_sessions' to get 'session_id'
        session_keys = ["season", "round", "session_type"]
        df = df.merge(dim_sessions[["session_id"] + session_keys],
                      on= session_keys,
                      how= "left")

        # Join with 'dim_drivers' to get 'driver_id' (match on 'driver_number', use the latest record for that season)
        # If 'dim_drivers' is not empty...
        if len(dim_drivers) > 0:
            driver_map = dim_drivers[["driver_id", "driver_number", "first_season_seen"]].copy() # Deep copy

            # Convert numeric columns to int/float
            if "first_season_seen" in driver_map.columns:
                driver_map["first_season_seen"] = driver_map["first_season_seen"].apply(self.silver_utils.str_to_int) # Int
            if "driver_id" in driver_map.columns:
                driver_map["driver_id"] = driver_map["driver_id"].apply(self.silver_utils.str_to_int) # Int

            # For each 'driver_number', keep the record with the highest 'first_season_seen' (smaller than lap's season)
            df = df.merge(driver_map.rename(columns= {"first_season_seen": "driver_season"}),
                          on= "driver_number",
                          how= "left")
            df = df[df["season"] >= df["driver_season"]].copy() # Deep copy 
            # Keep the most recent driver record per ('driver_number', 'season')
            df = df.sort_values("driver_season", ascending= False).drop_duplicates(subset= ["season", "round", "session_type", "driver_number", "lap_number"],
                                                                                   keep= "first")
        # Deduplicate by 'dedup_keys'
        dedup_keys = ["season", "round", "session_type", "driver_number", "lap_number"]
        before_len = len(df)
        df = df.drop_duplicates(subset= dedup_keys, keep="last")
        after_len = len(df)
        if before_len != after_len:
            print(f"[Info] Deduplicated: {before_len} -> {after_len} rows")

        # Select final columns
        final_cols = ["session_id", "driver_id", "driver_number", "lap_number", "stint_number", "lap_time_duration", "sector_1_time",
                      "sector_2_time", "sector_3_time", "speed_i1", "speed_i2", "speed_fl", "speed_st", "tire_compound", "tire_age",
                      "is_fresh_tire", "position", "team", "is_deleted", "deleted_reason", "is_accurate", "season", "round", "session_type"]
        available = [col for col in final_cols if col in df.columns] 
        final_df = df[available]

        # Data quallity checks
        self.silver_utils.assert_unique(final_df, ["session_id", "session_type", "driver_number", "lap_number"], "silver.fact_laps")  
        self.silver_utils.assert_range(final_df, "lap_time_duration", 30.0, 600.0, "silver.fact_laps")
        return final_df

    def build_fact_weather(self, bronze_weather_df, dim_sessions): # Builds 'fact_weather' table from Bronze's 'weather' table
        print("Building 'silver.fact_weather'...")
        # If 'bronze_weather_df' is empty...
        if len(bronze_weather_df) == 0:
            return pd.DataFrame()

        # Deep copy 'bronze_weather_df'
        df = bronze_weather_df.copy()

        # Type cast numeric columns
        numeric_cols = ["air_temp", "track_temp", "humidity", "pressure", "wind_direction", "wind_speed"]
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors= "coerce")
        # Convert numeric columns to int/float
        if "session_id" in df.columns:
            df["session_id"] = df["session_id"].apply(self.silver_utils.str_to_int) # Int
        if "season" in df.columns:
            df["season"] = df["season"].apply(self.silver_utils.str_to_int) # Int
        if "round" in df.columns:
            df["round"] = df["round"].apply(self.silver_utils.str_to_int) # Int

        # Join with 'dim_sessions' to get 'session_id'
        session_keys = ["season", "round", "session_type"]
        df = df.merge(dim_sessions[["session_id"] + session_keys],
                      on= session_keys,
                      how= "left")

        # Deduplicate by 'session_id' + 'weather_time'
        if "weather_time" in df.columns:
            df = df.drop_duplicates(subset= ["session_id", "weather_time"],
                                    keep= "last")

        # Select final columns
        final_cols = ["session_id", "weather_time", "air_temp", "track_temp", "humidity", "pressure", "wind_direction", "wind_speed",
                      "is_raining", "season", "round", "session_type"]
        available = [col for col in final_cols if col in df.columns]
        final_df = df[available]

        # Data quallity checks
        self.silver_utils.assert_range(final_df, "air_temp", -20, 60, "silver.fact_weather")
        self.silver_utils.assert_range(final_df, "track_temp", -20, 80, "silver.fact_weather")
        return final_df

    def build_fact_results(self, bronze_results_df, dim_sessions, dim_drivers): # Builds 'fact_results' table from Bronze's 'results' table 
        print("Building 'silver.fact_results'...")
        # If 'bronze_results_df' is empty...
        if len(bronze_results_df) == 0:
            return pd.DataFrame()
        # Deep copy 'bronze_results_df'
        df = bronze_results_df.copy()

        # Convert timedelta columns to integers (seconds)
        for col in ["q1_time", "q2_time", "q3_time", "total_time"]:
            if col in df.columns:
                df[col] = df[col].apply(self.silver_utils.str_to_int) # Int
        # Convert columns to integers
        if "position" in df.columns:
            df["position"] = df["position"].apply(self.silver_utils.str_to_int) # Int
        if "season" in df.columns:
            df["season"] = df["season"].apply(self.silver_utils.str_to_int) # Int
        if "round" in df.columns:
            df["round"] = df["round"].apply(self.silver_utils.str_to_int) # Int
        if "grid_position" in df.columns:
            df["grid_position"] = df["grid_position"].apply(self.silver_utils.str_to_int) # Int
        if "points" in df.columns:
            df["points"] = df["points"].apply(self.silver_utils.str_to_float) # Float

        # Join with 'dim_sessions'
        session_keys = ["season", "round", "session_type"]
        df = df.merge(dim_sessions[["session_id"] + session_keys],
                      on= session_keys,
                      how= "left")
        # Join with 'dim_drivers' to get 'driver_id'
        if len(dim_drivers) > 0: # If 'dim_drivers' is not empty...
            driver_map = dim_drivers[["driver_id", "driver_number"]].drop_duplicates(subset= ["driver_number"], 
                                                                                     keep= "first")
            df = df.merge(driver_map, 
                          on= "driver_number", 
                          how= "left")

        # Deduplicate by 'session_id' + 'driver_number'
        dedup_keys = ["session_id", "driver_number"]
        df = df.drop_duplicates(subset= dedup_keys, 
                                keep= "last")

        # Select final columns
        final_cols = ["session_id", "driver_id", "driver_number", "driver_code", "team", "position", "classified_position", "grid_position",
                      "q1_time", "q2_time", "q3_time", "total_time", "status", "points", "season", "round", "session_type"]
        available = [col for col in final_cols if col in df.columns]
        final_df = df[available]

        # Data quallity checks
        self.silver_utils.assert_unique(final_df, ["session_id", "driver_number"], "silver.fact_results")
        return final_df

    ## Write Methods
    def write_silver(self, df, table_name, partition_cols= None): # Writes DataFrame to silver Delta table
                                                                  # Uses OVERWRITE mode ('silver' is rebuilt from 'bronze' on each run)
        if len(df) == 0: # If empty...
            print(f"[Info] No data for silver.{table_name}, skipping.")
            return 0
        # Get table path
        path = self.catalog.config.table_path("silver", table_name)

        # Chunking for memory efficiency
        chunk_size = 25000
        for idx in range(0, len(df), chunk_size): # 'chunk_size': as range steps
            chunk_df = df.iloc[idx : idx + chunk_size]
            # Write first chunk as overwrite and append the rest
            write_mode = "overwrite" if idx == 0 else "append"
            # Creates the table (with overwrite)
            self.dl_wrapper.write_table(chunk_df, path, mode= write_mode, partition_cols= partition_cols or ["season"]) # 'partition_cols': if None/False, defaults to '["season"]'
        # Register in catalog
        self.catalog.register("silver", table_name,
                              partition_cols= partition_cols or ["season"],
                              description= f"Silver layer: {table_name}")
        print(f"[Info] Created silver.{table_name} with {len(df)} rows")
        return len(df)

    ## Pipeline Methods
    def run_silver(self): # Run bronze-to-silver pipeline (dimensions first, then facts)
        # Runs statistics placeholder
        stats = {}
        # Read all bronze tables
        print("Reading bronze tables...")
        bronze_sessions_df = self.dl_wrapper.read_table(catalog.get_path("bronze.sessions"))
        bronze_laps_df = self.dl_wrapper.read_table(catalog.get_path("bronze.laps"))
        bronze_results_df = self.dl_wrapper.read_table(catalog.get_path("bronze.results"))
        bronze_weather_df = self.dl_wrapper.read_table(catalog.get_path("bronze.weather"))

        # Build dimension tables
        print("[Info] Building silver dimensions...")
        dim_drivers = self.build_dim_drivers(bronze_results_df, bronze_laps_df)
        stats["dim_drivers"] = self.write_silver(dim_drivers, "dim_drivers", ["driver_id"])

        dim_constructors = self.build_dim_constructors(bronze_results_df)
        stats["dim_constructors"] = self.write_silver(dim_constructors, "dim_constructors", ["constructor_id"])

        dim_circuits = self.build_dim_circuits(bronze_sessions_df)
        stats["dim_circuits"] = self.write_silver(dim_circuits, "dim_circuits", ["circuit_id"])

        dim_sessions = self.build_dim_sessions(bronze_sessions_df, dim_circuits)
        stats["dim_sessions"] = self.write_silver(dim_sessions, "dim_sessions", ["session_id"])

        # Build fact tables
        print("[Info] Building silver facts...")
        fact_laps = self.build_fact_laps(bronze_laps_df, dim_sessions, dim_drivers)
        stats["fact_laps"] = self.write_silver(fact_laps, "fact_laps", ["session_id", "session_type", "driver_number", "lap_number"])  

        fact_weather = self.build_fact_weather(bronze_weather_df, dim_sessions)
        stats["fact_weather"] = self.write_silver(fact_weather, "fact_weather", ["session_id"])

        fact_results = self.build_fact_results(bronze_results_df, dim_sessions, dim_drivers)
        stats["fact_results"] = self.write_silver(fact_results, "fact_results", ["session_id", "driver_number"])

        print(f"Silver pipeline complete: {stats}")
        return stats


In [9]:
# Instantiating classes
silver_utils = SilverUtils()
silver_builder = SilverBuider(catalog, dl_wrapper, silver_utils)

In [27]:
# Running the silver pipeline
silver_builder.run_silver()

Reading bronze tables...
[Info] Building silver dimensions...
Building 'silver.dim_drivers'...
[Quality] Unique Pass: silver.dim_drivers unique by ['driver_id']
[Quality] NULLs Pass: silver.dim_drivers has no NULLs in ['driver_id', 'driver_number']
[Info] Created silver.dim_drivers with 38 rows
Building 'silver.dim_constructors'...
[Quality] Unique Pass: silver.dim_constructors unique by ['constructor_id']
[Info] Created silver.dim_constructors with 10 rows
Building 'silver.dim_circuits'...
[Quality] Unique Pass: silver.dim_circuits unique by ['circuit_id']
[Info] Created silver.dim_circuits with 24 rows
Building 'silver.dim_sessions'...
[Quality] Unique Pass: silver.dim_sessions unique by ['session_id']
[Quality] NULLs Pass: silver.dim_sessions has no NULLs in ['session_id', 'season', 'round', 'session_type']
[Info] Created silver.dim_sessions with 108 rows
[Info] Building silver facts...
Building 'silver.fact_laps'...
[Quality] Unique Pass: silver.fact_laps unique by ['session_id', '

{'dim_drivers': 38,
 'dim_constructors': 10,
 'dim_circuits': 24,
 'dim_sessions': 108,
 'fact_laps': 62131,
 'fact_weather': 10997,
 'fact_results': 2159}

**8) Silver to Gold Transformations**<br>
We read ```silver``` tables (from ```star schema```), compute analytical aggregated tables, and write the result to ```Gold```.

In [13]:
class GoldBuider:
    def __init__(self, catalog, dl_wrapper):
        # Config
        self.catalog = catalog
        # Wrapper
        self.dl_wrapper = dl_wrapper
        
    ## Dimension Tables Builders 
    def build_gold_dim_drivers(self, silver_drivers_df): # Builds 'dim_drivers' table from Silver's 'dim_drivers' table 
                                                    # Gold layer is a cleaned, consumer-ready version with only current SCD records (plus flat structure)
        print("Building 'gold.dim_drivers'...")
        # Deep copy of Silver (already cleansed)
        df = silver_drivers_df.copy()
        # Add 'display_name' column for convenience
        df["display_name"] = df["full_name"]
        # Add 'is_active' flag (drivers seen in the most recent season are active)
        if "first_season_seen" in df.columns: 
            max_season = df["first_season_seen"].max()
            df["is_active"] = df["first_season_seen"] == max_season # If current season equals to latest season, driver is active
        else:
            df["is_active"] = True # If not on table already, driver is in first season (active)
        return df
    
    def build_gold_dim_circuits(self, silver_circuits_df): # Builds 'dim_circuits' table from Silver's 'dim_circuits' table 
                                                           # Pass through without enrichment
        print("Building 'gold.dim_circuits'...")
        return silver_circuits_df.copy() # Deep copy

    def build_gold_dim_constructors(self, silver_constructors_df): # Builds 'dim_constructors' table from Silver's 'dim_constructors' table  
                                                                   # Pass through without enrichment    
        print("Building 'gold.dim_constructors'...")
        return silver_constructors_df.copy()
    
    def build_gold_dim_sessions(self, silver_sessions_df): # Builds 'dim_sessions' table from Silver's 'dim_sessions' table  
                                                           # Adds convenience columns for querying
        print("Building 'gold.dim_sessions'...")
        # Deep copy of Silver (already cleansed)
        df = silver_sessions_df.copy()
        # Add human-readable session label (ie "2024 R01 - Race") 
        df["session_label"] = (df["season"].astype(str) + " Round" + df["round"].astype(str).str.zfill(2) + " - " + df["session_name"]) # 'zfill()': pads string with leading zeros 
        return df
    
    ## Fact Tables Builders 
    def build_gold_fact_lap_times(self, silver_laps_df): # Builds 'fact_lap_times' table from Silver's 'fact_laps' table
                                                         # This is the primary fact table for lap-level analysis (one row per driver per lap per session)
        print("Building 'gold.fact_lap_times'...")
        # Deep copy of Silver (already cleansed)
        df = silver_laps_df.copy()
        # Filter out deleted laps (not valid for analysis) 
        if "is_deleted" in df.columns: 
            df = df[~df["is_deleted"].map({"false": False, "true": True}).fillna(False)] # Remove deleted and NA rows
                                                                                         # 'fillna(False)': returns a string thus 'astype(bool)' to cast to bool
        # Rename column (for schema clarity)
        df = df.rename(columns={"lap_time_duration": "lap_time_sec"})

        ## Add calculated columns
        # Lap time rank within the session (1: fastest lap in the session)
        df["lap_rank_session"] = df.groupby(["session_id", "lap_number"])["lap_time_sec"].rank() # Rank at session-lap level
        # Is this lap the fastest lap in the session?
        min_lap_time = df.groupby("session_id")["lap_time_sec"].transform("min")
        df["is_fastest_lap"] = df["lap_time_sec"] == min_lap_time # Bool
        # Position change from previous lap (gained/lost positions)
        df = df.sort_values(["session_id", "driver_id", "lap_number"])
        df["prev_position"] = df.groupby(["session_id", "driver_id"])["position"].shift(1)
        df["positions_change"] = df["prev_position"] - df["position"]
        return df
    
    def build_gold_fact_pit_stops(self, silver_laps_df): # Builds 'fact_pit_stops' table from Silver's 'fact_laps' table 
                                                         # 'stint': number of continuous laps a driver spends on track between pit stops
                                                         # Derives pit stop events by detecting when a driver's stint number changes or when pit_in/pit_out times are present (one row pit stop per driver per session)
        print("Building 'gold.fact_pit_stops'...")
        # Deep copy of Silver (already cleansed)
        df = silver_laps_df.copy()
        df = df.sort_values(["session_id", "driver_number", "lap_number"]) # Sort subset columns
        # Placeholder
        pit_stops = []
        # Detect pit stops by looking at stint changes
        for (session_id, driver_id), group in df.groupby(["session_id", "driver_number"]): # '(session_id, driver_id)': columns subset to group
                                                                                           # 'group': DataFrame with only the rows that belong to subset (above)
            prev_stint = None
            for idx, row in group.iterrows():
                current_stint = row.get("stint_number")
                if prev_stint is not None and current_stint != prev_stint:
                    # Stint changed: pit stop happened between these laps
                    pit_stops.append({"session_id": session_id,
                                      "driver_id": driver_id,
                                      "lap_number": row.get("lap_number"),
                                      "stint_from": prev_stint,
                                      "stint_to": current_stint,
                                      "tire_compound": row.get("tire_compound"),
                                      "tire_age": row.get("tire_age")})
                prev_stint = current_stint
        # If no pit stops...
        if not pit_stops:
            return pd.DataFrame()
        # Builds pit stops DataFrame
        result = pd.DataFrame(pit_stops)
        # Add pit stop sequence number per driver per session
        result = result.sort_values(["session_id", "driver_id", "lap_number"])
        result["pit_stop_number"] = result.groupby(["session_id", "driver_id"]).cumcount() + 1 # Counts 'result' by 'session_id'/'driver_id'
        return result

    def build_gold_fact_race_results(self, silver_results_df, silver_sessions_df): # Builds 'fact_race_results' table from Silver's 'fact_results' table  
                                                                                   # Filters race sessions only (R) and adds derived metrics (like positions gained/lost)
        print("Building 'gold.fact_race_results'...")
        # Deep copy of Silver (already cleansed)
        df = silver_results_df.copy() 
        # Filter to race sessions only
        if "session_type" in df.columns:
            df = df[df["session_type"] == "R"]

        # Join with sessions to get event context
        session_cols = ["session_id", "season", "round", "event_name", "circuit_name"] 
        available_sessions = [col for col in session_cols if col in silver_sessions_df.columns]
        # If columns are present... 
        if available_sessions:
            df = df.merge(silver_sessions_df[available_sessions], on= "session_id", how= "left", suffixes= ("", "_session")) # 'merge()': suffixes to add to overlapping col names in left/right respectively

        # Calculate positions gained/lost from grid
        if "grid_position" in df.columns and "position" in df.columns:
            df["positions_gained"] = df["grid_position"] - df["position"] # 'positions_gained': considers only starting grid position
        # Is podium finish?
        df["is_podium"] = df["position"].isin([1, 2, 3]) # Bool
        # Is Did Not Finish (DNF)?
        if "status" in df.columns:
            df["is_dnf"] = df["status"].str.contains("Retired | DNF | Accident | Collision | Engine | Gearbox | Transmission | Hydraulic | Electrical | Power", case= False, na= False)
        # Is win?
        df["is_win"] = df["position"] == 1 # Bool
        return df

    ## Aggregate Tables Builders
    def build_agg_driver_performance(self, gold_lap_times_df, gold_race_results_df): # Builds 'agg_driver_performance' table from Gold's 'fact_race_results'/'fact_lap_times' tables  
                                                                                     # Per-driver, per-race performance metrics (such as fastest lap, avg lap, etc)
        print("[Info] Building 'gold.agg_driver_performance'...")
        # Aggregate lap times per driver per session (with respective functions)
        lap_agg = gold_lap_times_df.groupby(["session_id", "driver_number"]).agg(avg_lap_time_sec= ("lap_time_sec", "mean"),
                                                                                 fastest_lap_time_sec= ("lap_time_sec", "min"),
                                                                                 lap_time_std_sec= ("lap_time_sec", "std"),
                                                                                 total_laps= ("lap_number", "count"),
                                                                                 avg_position= ("position", "mean"),
                                                                                 best_position= ("position", "min")).reset_index()

        # Merge with race results for final position/points
        if len(gold_race_results_df) > 0:
            results_cols = ["session_id", "driver_number", "position", "points", "grid_position", "positions_gained", "is_podium", "is_win", "is_dnf"]
            available_race = [col for col in results_cols if col in gold_race_results_df.columns]
            lap_agg = lap_agg.merge(gold_race_results_df[available_race], on= ["session_id", "driver_number"], how= "left")

        # Calculate lap time consistency as coefficient of variation (lower= more consistent)
        lap_agg["lap_time_cv"] = lap_agg["lap_time_std_sec"] / lap_agg["avg_lap_time_sec"]

        # Rolling form: average finishing position over last 5 races (per driver)
        if "position" in lap_agg.columns:
            lap_agg = lap_agg.sort_values(["driver_number", "session_id"])
            lap_agg["rolling_avg_position_5"] = (lap_agg.groupby("driver_number")["position"].rolling(window= 5, min_periods= 1)
                                                                                             .mean()
                                                                                             .reset_index(level= 0, drop= True))
            lap_agg["rolling_avg_points_5"] = (lap_agg.groupby("driver_number")["points"].rolling(window= 5, min_periods= 1)
                                                                                         .mean()
                                                                                         .reset_index(level= 0, drop= True))
        return lap_agg 

    def build_agg_constructor_reliability(self, gold_race_results_df): # Builds 'agg_constructor_reliability' table from Gold's 'fact_race_results' tables
                                                                       # Per-constructor, per-season reliability metrics
        print("Building 'gold.agg_constructor_reliability'...")
        # If empty...
        if len(gold_race_results_df) == 0:
            return pd.DataFrame()
        # Deep copy of Silver (already cleansed)
        df = gold_race_results_df.copy()

        # Group by team and season (with respective functions)
        agg = df.groupby(["team", "season"]).agg(total_races= ("session_id", "count"),
                                                 total_dnfs= ("is_dnf", "sum"),
                                                 total_wins= ("is_win", "sum"),
                                                 total_podiums= ("is_podium", "sum"),
                                                 total_points= ("points", "sum"),
                                                 avg_position= ("position", "mean"),
                                                 avg_grid_position= ("grid_position", "mean")).reset_index()
        # Calculate rates
        agg["dnf_rate"] = agg["total_dnfs"] / agg["total_races"] # Did Not Finish
        agg["win_rate"] = agg["total_wins"] / agg["total_races"]
        agg["podium_rate"] = agg["total_podiums"] / agg["total_races"]
        return agg

    ## Write Methods
    def write_gold(self, df, table_name, partition_cols= None): # Writes DataFrame to gold Delta table
        if len(df) == 0: # If empty...
            print(f"[Info] No data for gold.{table_name}, skipping.")
            return 0
        # Get table path
        path = self.catalog.config.table_path("gold", table_name)

        # Chunking for memory efficiency
        chunk_size = 50000
        for idx in range(0, len(df), chunk_size): # 'chunk_size': as range steps
            chunk_df = df.iloc[idx : idx + chunk_size]
            # Write first chunk as overwrite and append the rest
            write_mode = "overwrite" if idx == 0 else "append"
            # Creates the table (with overwrite)
            self.dl_wrapper.write_table(df, path, mode= write_mode, partition_cols= partition_cols or ["season"]) # 'partition_cols': if None/False, defaults to '["season"]'
        # Register in catalog
        self.catalog.register("gold", table_name,
                              partition_cols= partition_cols or ["season"],
                              description= f"Gold layer: {table_name}")
        print(f"[Info] Created gold.{table_name} with {len(df)} rows")
        return len(df)

    ## Pipeline Methods
    def run_gold(self): # Run silver-to-gold pipeline (dimensions first, then facts, then aggregates)
        # Runs statistics placeholder
        stats = {}

        # Read all bronze tables
        print("Reading silver tables...")
        silver_drivers = self.dl_wrapper.read_table(catalog.get_path("silver.dim_drivers"))
        silver_constructors = self.dl_wrapper.read_table(catalog.get_path("silver.dim_constructors"))
        silver_circuits = self.dl_wrapper.read_table(catalog.get_path("silver.dim_circuits"))
        silver_sessions = self.dl_wrapper.read_table(catalog.get_path("silver.dim_sessions"))
        silver_laps = self.dl_wrapper.read_table(catalog.get_path("silver.fact_laps"))
        silver_results = self.dl_wrapper.read_table(catalog.get_path("silver.fact_results"))

        # Build dimension tables
        print("[Info] Building gold dimensions...")
        gold_drivers = self.build_gold_dim_drivers(silver_drivers)
        stats["dim_drivers"] = self.write_gold(gold_drivers, "dim_drivers", ["driver_id"]) 

        gold_circuits = self.build_gold_dim_circuits(silver_circuits)
        stats["dim_circuits"] = self.write_gold(gold_circuits, "dim_circuits", ["circuit_id"]) 

        gold_constructors = self.build_gold_dim_constructors(silver_constructors)
        stats["dim_constructors"] = self.write_gold(gold_constructors, "dim_constructors", ["constructor_id"]) 

        gold_sessions = self.build_gold_dim_sessions(silver_sessions)
        stats["dim_sessions"] = self.write_gold(gold_sessions, "dim_sessions", ["session_id"]) 

        # Build fact tables
        print("[Info] Building gold facts...")
        gold_lap_times = self.build_gold_fact_lap_times(silver_laps)
        stats["fact_lap_times"] = self.write_gold(gold_lap_times, "fact_lap_times", ["session_id"]) 

        gold_pit_stops = self.build_gold_fact_pit_stops(silver_laps)
        stats["fact_pit_stops"] = self.write_gold(gold_pit_stops, "fact_pit_stops", ["session_id"]) 

        gold_race_results = self.build_gold_fact_race_results(silver_results, silver_sessions)
        stats["fact_race_results"] = self.write_gold(gold_race_results, "fact_race_results", ["session_id"]) 

        # Build gold aggregates
        print("Building gold aggregates...")
        agg_driver_perf = self.build_agg_driver_performance(gold_lap_times, gold_race_results)
        stats["agg_driver_performance"] = self.write_gold(agg_driver_perf, "agg_driver_performance", ["session_id"])

        agg_constructor_rel = self.build_agg_constructor_reliability(gold_race_results)
        stats["agg_constructor_reliability"] = self.write_gold(agg_constructor_rel, "agg_constructor_reliability", ["season"])

        print(f"Gold pipeline complete: {stats}")
        return stats

In [14]:
# Instantiating classes
gold_builder = GoldBuider(catalog, dl_wrapper)

In [15]:
# Running the gold pipeline
gold_builder.run_gold()

Reading silver tables...
[Info] Building gold dimensions...
Building 'gold.dim_drivers'...
[Info] Created gold.dim_drivers with 38 rows
Building 'gold.dim_circuits'...
[Info] Created gold.dim_circuits with 24 rows
Building 'gold.dim_constructors'...
[Info] Created gold.dim_constructors with 10 rows
Building 'gold.dim_sessions'...
[Info] Created gold.dim_sessions with 108 rows
[Info] Building gold facts...
Building 'gold.fact_lap_times'...
[Info] Created gold.fact_lap_times with 62131 rows
Building 'gold.fact_pit_stops'...
[Info] Created gold.fact_pit_stops with 15147 rows
Building 'gold.fact_race_results'...
[Info] Created gold.fact_race_results with 479 rows
Building gold aggregates...
[Info] Building 'gold.agg_driver_performance'...
[Info] Created gold.agg_driver_performance with 2152 rows
Building 'gold.agg_constructor_reliability'...
[Info] Created gold.agg_constructor_reliability with 10 rows
Gold pipeline complete: {'dim_drivers': 38, 'dim_circuits': 24, 'dim_constructors': 10, '

{'dim_drivers': 38,
 'dim_circuits': 24,
 'dim_constructors': 10,
 'dim_sessions': 108,
 'fact_lap_times': 62131,
 'fact_pit_stops': 15147,
 'fact_race_results': 479,
 'agg_driver_performance': 2152,
 'agg_constructor_reliability': 10}

---